# WaferGuard — PyTorch supervised autoencoder audit

**Decision: NO-GO for production based on the currently documented evidence.** This notebook uses a chronological train/validation/test split, training-only preprocessing, natural-prevalence evaluation, and an explicit validation-only threshold choice. Run all cells from a clean kernel to generate fresh metrics. No result is asserted before that run.

## Audit and repair notes

This notebook is designed to make the relevant safeguards executable and auditable:

1. **Leakage:** rows are ordered by `timestamp` before splitting; imputation and scaling are fitted on the training partition only. SMOTE is not used, so synthetic samples cannot cross partition boundaries and evaluation retains natural prevalence.
2. **Reconstruction-loss shape:** model inputs are consistently `[batch, features]`; reconstruction targets have the same shape. An assertion checks this before calculating MSE, preventing accidental broadcasting such as `[B, 1, F]` against `[B, F]`.
3. **Legacy PS-L encoding:** `sensor_0 == 0.0` is treated as a legacy sentinel, not a continuous measurement. `sensor_1` values 100/200/300 are represented as categorical indicators. The raw `sensor_0` and `sensor_1` columns are excluded from continuous imputation and scaling.
4. **Evaluation:** the threshold is selected using validation predictions only. Test metrics and bootstrap confidence intervals are calculated once from the held-out chronological test partition. They must be generated by running this notebook; no stale output is treated as evidence.

The dataset is expected at `data/raw/secom_telemetry_modified.csv` or as `secom_telemetry_modified.csv` beside the notebook.

## 0 · Environment

Install the pinned dependencies once from the repository root, outside the notebook: `pip install -r requirements.txt`.

In [ ]:
import os, sys, random, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix
)
import joblib

# Locate the repository root whether the kernel starts there or in notebooks/.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The model class and split definition are shared with the Streamlit app and scripts.
from src.models.wafer_model import WaferAutoencoder
from src.data.splits import (
    TRAIN_FRACTION, VALIDATION_FRACTION, sort_chronologically, split_bounds
)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=128, latent=32, lambda_clf=1.0,
          train_fraction=TRAIN_FRACTION, validation_fraction=VALIDATION_FRACTION)
print(f"torch {torch.__version__} | device={DEVICE}")

## 1. Data


In [ ]:
candidate_paths = [
    os.path.join("data", "raw", "secom_telemetry_modified.csv"),
    "secom_telemetry_modified.csv",
]
data_path = next((p for p in candidate_paths if os.path.isfile(p)), None)
if data_path is None:
    raise FileNotFoundError(
        "Place secom_telemetry_modified.csv in data/raw/ or beside this notebook."
    )

df = pd.read_csv(data_path)
SENSOR_COLS = sorted(
    [c for c in df.columns if c.startswith("sensor_")],
    key=lambda c: int(c.split("_")[1])
)
required = {"label", "timestamp", "sensor_0", "sensor_1"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
if not SENSOR_COLS:
    raise ValueError("No sensor_* columns were found.")

# Use chronological order to ensure later lots cannot influence earlier predictions.
# The stable sort and the split below are shared with the app (src/data/splits.py).
df = sort_chronologically(df)

y = (df["label"].to_numpy() == 1).astype(np.int64)
legacy_sentinel = df["sensor_0"].to_numpy(dtype=np.float64) == 0.0
sensor1 = df["sensor_1"].to_numpy(dtype=np.float64)
# Same definition as src/services/prediction_service.py: sensor_1 codes only count
# on legacy (sensor_0 == 0) rows, so training and serving build identical features.
legacy_code_100 = legacy_sentinel & (sensor1 == 100.0)
legacy_code_200 = legacy_sentinel & (sensor1 == 200.0)
legacy_code_300 = legacy_sentinel & (sensor1 == 300.0)

# Keep sentinel/code semantics outside the continuous imputer and scaler.
continuous_cols = [c for c in SENSOR_COLS if c not in {"sensor_0", "sensor_1"}]
FLAG_COLS = [
    "is_legacy_ps_l",
    "legacy_sensor_1_code_100",
    "legacy_sensor_1_code_200",
    "legacy_sensor_1_code_300",
]
X_cont_raw = df[continuous_cols].to_numpy(dtype=np.float64)
legacy_flags = np.column_stack([
    legacy_sentinel, legacy_code_100, legacy_code_200, legacy_code_300
]).astype(np.float64)
print(f"Rows: {len(df)} | continuous sensors: {len(continuous_cols)}")
print(f"Natural defect prevalence: {y.mean():.2%} ({int(y.sum())} defects / {int((1-y).sum())} non-defects)")
print(f"Missing continuous cells: {np.isnan(X_cont_raw).mean():.2%}")
print(f"sensor_0 == 0 sentinel rows: {int(legacy_sentinel.sum())}")
print(f"sensor_1 categorical code counts: 100={int(legacy_code_100.sum())}, "
      f"200={int(legacy_code_200.sum())}, 300={int(legacy_code_300.sum())}")

## 2 · Preprocessing & class rebalancing


In [ ]:
# Chronological split: earliest rows train, following rows validate, latest rows test.
n = len(df)
train_end, validation_end = split_bounds(n, HP["train_fraction"], HP["validation_fraction"])
i_train = np.arange(0, train_end)
i_val = np.arange(train_end, validation_end)
i_test = np.arange(validation_end, n)

X_train_raw, X_val_raw, X_test_raw = X_cont_raw[i_train], X_cont_raw[i_val], X_cont_raw[i_test]
F_train, F_val, F_test = legacy_flags[i_train], legacy_flags[i_val], legacy_flags[i_test]
y_train, y_val, y_test = y[i_train], y[i_val], y[i_test]

# Fit both transformations on training data only. The imputer is fitted on a named
# frame so the app can transform raw[usable_cols] without feature-name warnings.
imputer = SimpleImputer(strategy="median", keep_empty_features=True)
X_train_imp = imputer.fit_transform(pd.DataFrame(X_train_raw, columns=continuous_cols))
X_val_imp = imputer.transform(pd.DataFrame(X_val_raw, columns=continuous_cols))
X_test_imp = imputer.transform(pd.DataFrame(X_test_raw, columns=continuous_cols))

scaler = StandardScaler()
X_train_cont = scaler.fit_transform(X_train_imp)
X_val_cont = scaler.transform(X_val_imp)
X_test_cont = scaler.transform(X_test_imp)

# Append unscaled semantic indicators; do not impute/scale the sentinel/categorical columns.
X_train = np.concatenate([X_train_cont, F_train], axis=1).astype(np.float32)
X_val = np.concatenate([X_val_cont, F_val], axis=1).astype(np.float32)
X_test = np.concatenate([X_test_cont, F_test], axis=1).astype(np.float32)

print(f"chronological split: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")
print(f"defect prevalence: train={y_train.mean():.2%}, validation={y_val.mean():.2%}, test={y_test.mean():.2%}")
print(f"split boundaries: train ends {df.loc[i_train[-1], 'timestamp']}, "
      f"validation ends {df.loc[i_val[-1], 'timestamp']}, test ends {df.loc[i_test[-1], 'timestamp']}")

## 3 · Tensors & loaders


In [ ]:
F = X_train.shape[1]

def make_loader(X, y, shuffle):
    Xt = torch.from_numpy(np.asarray(X, dtype=np.float32))  # [N, F]
    yt = torch.from_numpy(np.asarray(y, dtype=np.float32))  # [N]
    assert Xt.ndim == 2 and Xt.shape[0] == yt.shape[0]
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(
        TensorDataset(Xt, yt), batch_size=HP["batch_size"],
        shuffle=shuffle, generator=generator, drop_last=False
    )

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)
print(f"features={F} | train batches={len(train_loader)} | validation batches={len(val_loader)} | test batches={len(test_loader)}")

## 4 · Model — supervised autoencoder

The architecture lives in `src/models/wafer_model.py` (`WaferAutoencoder`) so the notebook and the Streamlit app train and serve the same class. Its forward pass raises if reconstruction or logit shapes do not match the input.

In [ ]:
model = WaferAutoencoder(
    F, latent_dim=HP["latent"], hidden_dim=HP["hidden"]
).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5 · Training


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
n_pos = max(int(y_train.sum()), 1)
n_neg = max(int((1 - y_train).sum()), 1)
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

history = {"total": [], "recon": [], "clf": [], "val_ap": [], "val_auc": []}
best_val_ap = -np.inf
best_epoch = 0
best_state = None
patience = 6
epochs_without_improvement = 0

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    totals = dict(total=0.0, recon=0.0, clf=0.0)
    n_seen = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        reconstructed, logits, _ = model(batch_x)
        assert reconstructed.shape == batch_x.shape
        assert logits.shape == batch_y.shape
        recon_loss = torch.mean((reconstructed - batch_x) ** 2)
        clf_loss = bce(logits, batch_y)
        loss = recon_loss + HP["lambda_clf"] * clf_loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        bs = batch_x.shape[0]
        totals["total"] += loss.item() * bs
        totals["recon"] += recon_loss.item() * bs
        totals["clf"] += clf_loss.item() * bs
        n_seen += bs
    sched.step()
    for k in totals:
        history[k].append(totals[k] / n_seen)

    # Early stopping uses validation ranking quality only; test data is not consulted.
    model.eval()
    val_score_chunks = []
    with torch.no_grad():
        for xb, _ in val_loader:
            _, logits, _ = model(xb.to(DEVICE))
            val_score_chunks.append(torch.sigmoid(logits).cpu().numpy())
    epoch_val_probs = np.concatenate(val_score_chunks)
    val_ap_epoch = average_precision_score(y_val, epoch_val_probs) if y_val.sum() else float("nan")
    val_auc_epoch = roc_auc_score(y_val, epoch_val_probs) if len(np.unique(y_val)) == 2 else float("nan")
    history["val_ap"].append(float(val_ap_epoch))
    history["val_auc"].append(float(val_auc_epoch))

    if np.isfinite(val_ap_epoch) and val_ap_epoch > best_val_ap:
        best_val_ap = float(val_ap_epoch)
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch {epoch:02d}/{HP['epochs']} | total={history['total'][-1]:.4f} "
            f"| recon={history['recon'][-1]:.4f} | clf={history['clf'][-1]:.4f} "
            f"| val_AP={val_ap_epoch:.4f} | val_AUC={val_auc_epoch:.4f}"
        )
    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}; best validation AP={best_val_ap:.4f} at epoch {best_epoch}.")
        break

if best_state is None:
    raise RuntimeError("No valid validation AP was produced; model checkpoint was not selected.")
model.load_state_dict(best_state)
print(f"Restored best validation-AP checkpoint from epoch {best_epoch} (validation AP={best_val_ap:.4f}).")

## 6 · Evaluation


In [ ]:
def predict_probabilities(loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for xb, _ in loader:
            _, logits, _ = model(xb.to(DEVICE))
            chunks.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(chunks)

val_probs = predict_probabilities(val_loader)
test_probs = predict_probabilities(test_loader)

# Report validation ranking metrics. Threshold selection is independent of test labels.
val_auc = roc_auc_score(y_val, val_probs) if len(np.unique(y_val)) == 2 else float("nan")
val_ap = average_precision_score(y_val, val_probs) if y_val.sum() else float("nan")

# Use a validation-derived 10% alert-rate policy instead of maximizing F1 near zero.
TARGET_ALERT_RATE = 0.10
threshold = float(np.quantile(val_probs, 1.0 - TARGET_ALERT_RATE))
val_preds = (val_probs >= threshold).astype(int)
preds = (test_probs >= threshold).astype(int)

def report_metrics(y_true, scores, predictions, prefix):
    f1 = f1_score(y_true, predictions, zero_division=0)
    precision = precision_score(y_true, predictions, zero_division=0)
    recall = recall_score(y_true, predictions, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    auc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else float("nan")
    ap = average_precision_score(y_true, scores) if y_true.sum() else float("nan")
    print(f"{prefix}: n={len(y_true)} | defects={int(np.sum(y_true))} | prevalence={np.mean(y_true):.2%}")
    print(f"{prefix}: ROC-AUC={auc:.4f} | AP={ap:.4f} | precision={precision:.4f} | recall={recall:.4f} | F1={f1:.4f}")
    print(f"{prefix}: alert rate={np.mean(predictions):.2%} | TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    return {"f1": f1, "precision": precision, "recall": recall,
            "tn": tn, "fp": fp, "fn": fn, "tp": tp, "auc": auc, "ap": ap}

print(f"Best-checkpoint validation AP={best_val_ap:.4f} at epoch {best_epoch}")
print(f"Validation ROC-AUC={val_auc:.4f} | validation AP={val_ap:.4f}")
print(f"Threshold selected from validation scores for nominal {TARGET_ALERT_RATE:.0%} alert rate: {threshold:.8g}")
print(f"Actual validation alert rate (ties may increase it): {np.mean(val_preds):.2%}")
val_results = report_metrics(y_val, val_probs, val_preds, "VALIDATION")
test_results = report_metrics(y_test, test_probs, preds, "TEST — exploratory because test data was previously inspected")

# Bootstrap intervals for ranking metrics; this does not undo prior test-set inspection.
rng = np.random.default_rng(SEED)
auc_boot, ap_boot = [], []
if len(np.unique(y_test)) == 2 and y_test.sum() >= 2 and (1-y_test).sum() >= 2:
    for _ in range(2000):
        idx = rng.integers(0, len(y_test), size=len(y_test))
        if len(np.unique(y_test[idx])) < 2:
            continue
        auc_boot.append(roc_auc_score(y_test[idx], test_probs[idx]))
        ap_boot.append(average_precision_score(y_test[idx], test_probs[idx]))

def ci95(values):
    return tuple(np.percentile(values, [2.5, 97.5])) if len(values) else (float("nan"), float("nan"))

auc_ci = ci95(auc_boot)
ap_ci = ci95(ap_boot)
print(f"TEST ROC-AUC bootstrap 95% CI: [{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]")
print(f"TEST AP bootstrap 95% CI: [{ap_ci[0]:.4f}, {ap_ci[1]:.4f}]")

## 7 · Release decision

**Decision: NO-GO for production.** Use validation-only model selection and the validation-derived nominal 10% alert-rate threshold. Because the test set has previously been inspected, test metrics remain exploratory rather than an untouched final estimate. A low average precision near the natural prevalence and poor defect detection would not support deployment; use the fresh output from the current run when completing `AUDIT.md`.

In [ ]:
# Placeholder acceptance gate (set the real values with the domain owners). The alert
# policy must concentrate defects well above the natural defect rate, and the ranking
# quality must be distinguishable from chance: the bootstrap lower bound of test
# average precision has to exceed the natural prevalence. Lift alone is not enough
# with so few test defects.
MIN_LIFT = 2.0
test_prevalence = float(np.mean(y_test))
test_alert_rate = float(np.mean(preds))
test_recall = float(test_results["recall"])
test_lift = float(test_results["precision"]) / test_prevalence if test_prevalence else float("nan")
lift_ok = bool(np.isfinite(test_lift) and test_lift >= MIN_LIFT)
ap_ok = bool(np.isfinite(ap_ci[0]) and ap_ci[0] > test_prevalence)
verdict = "MET — requires operational review" if (lift_ok and ap_ok) else "NOT MET"
print(f"Test alert rate {test_alert_rate:.2%} | recall at that alert rate {test_recall:.2%} | "
      f"lift over prevalence {test_lift:.2f}x (need ≥ {MIN_LIFT:.1f}x: {'ok' if lift_ok else 'no'})")
print(f"Test AP bootstrap lower bound {ap_ci[0]:.4f} vs prevalence {test_prevalence:.4f} "
      f"({'ok' if ap_ok else 'no'})")
print(f"Reference gate: {verdict}")
print("Release decision: NO-GO pending evidence of useful performance and operational validation.")

import shutil
from datetime import datetime

# Save in the exact format src/services/prediction_service.py loads, so the
# Streamlit app and the evaluation scripts can use this run directly. Set
# WAFERGUARD_ARTIFACT_DIR to write somewhere other than <repo>/artifacts.
ARTIFACT_DIR = Path(os.environ.get("WAFERGUARD_ARTIFACT_DIR", REPO_ROOT / "artifacts"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = ARTIFACT_DIR / "wafer_model.pt"
PREPROCESSOR_FILE = ARTIFACT_DIR / "preprocessor.joblib"

# Keep a timestamped copy of any artifact this run is about to replace.
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
for existing in (MODEL_FILE, PREPROCESSOR_FILE):
    if existing.exists():
        backup = existing.with_name(f"{existing.stem}.{stamp}{existing.suffix}.bak")
        shutil.copy2(existing, backup)
        print(f"Backed up {existing.name} -> {backup.name}")

torch.save(
    {
        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "input_dim": int(X_train.shape[1]),
        "latent_dim": int(HP["latent"]),
        "hidden_dim": int(HP["hidden"]),
        "threshold": float(threshold),
        "usable_cols": list(continuous_cols),
        "flag_cols": list(FLAG_COLS),
        "target_alert_rate": float(TARGET_ALERT_RATE),
        "best_epoch": int(best_epoch),
        "best_validation_ap": float(best_val_ap),
    },
    str(MODEL_FILE),
)
joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "sensor_cols": list(SENSOR_COLS),
        "usable_cols": list(continuous_cols),
        "flag_cols": list(FLAG_COLS),
        "feature_cols": list(continuous_cols) + list(FLAG_COLS),
    },
    str(PREPROCESSOR_FILE),
)
print(f"Saved model checkpoint: {MODEL_FILE}")
print(f"Saved preprocessor: {PREPROCESSOR_FILE}")
print(f"Feature count: {X_train.shape[1]} | threshold: {threshold:.8g} | best epoch: {best_epoch}")

# Train/serve parity: reload the saved files through the app's own inference code
# and require the same features, scores and alerts as this notebook produced.
from src.services.prediction_service import load_model, prepare_features, predict_anomaly

app_model, app_preprocessor, app_threshold = load_model(MODEL_FILE, PREPROCESSOR_FILE)
app_features = prepare_features(df.iloc[i_test], app_preprocessor)
app_result = predict_anomaly(app_model, app_features, app_threshold)
assert np.allclose(app_features, X_test, atol=1e-5), "App features differ from notebook features."
assert np.allclose(app_result["probabilities"], test_probs, atol=1e-5), "App scores differ from notebook scores."
assert np.array_equal(np.asarray(app_result["predictions"]), preds), "App alerts differ from notebook alerts."
print("Artifact parity check passed: the app reproduces this notebook's test features, scores and alerts.")